## **1. Data Load**

In [1]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import nltk
from nltk.tokenize import sent_tokenize

nltk.download('punkt', quiet=True)

df = pd.read_csv("../1_data/processed/03_vader_scores.csv")
analyzer = SentimentIntensityAnalyzer()
print("Data loaded successfully!")
print(f"Shape: {df.shape}")

Data loaded successfully!
Shape: (22980, 21)


## **2. Define Aspects for Set C**

In [2]:
aspect_keywords = {
    'seat': ['seat', 'legroom', 'comfort', 'recline', 'space', 'cushion', 'comfy', 'spacious', 'stretch'],
    'food': ['food', 'meal', 'drink', 'snack', 'beverage', 'dining', 'menu', 'vegetarian'],
    'staff': ['crew', 'staff', 'attendant', 'stewardess', 'service', 'friendly', 'rude', 'attentive'],
    'ground_service': ['delay', 'late', 'on time', 'punctual', 'schedule', 'depart', 'cancel', 'luggage', 'suitcase',
                       'baggage', 'lost', 'checkin', 'check-in', 'refund', 'booking', 'boarding'],
    'entertainment': ['entertainment', 'screen', 'movie', 'ife', 'wifi', 'music']  # ife stands for In-Flight Entertainment
}

for aspect, keywords in aspect_keywords.items():
    count = df['cleaned_review_C'].str.contains('|'.join(keywords), case=False, na=False).sum()
    print(f"{aspect}: {count} ({count/len(df)*100:.1f}%)")

seat: 9281 (40.4%)
food: 8809 (38.3%)
staff: 16778 (73.0%)
ground_service: 16545 (72.0%)
entertainment: 4894 (21.3%)


> ### **Rationale for Selected Aspects**
>
> Five service dimensions were selected as the aspect categories for rule-based sentiment extraction: **seat, food, staff, ground_service, and entertainment**. All five aspects appear in a meaningful proportion of reviews, with entertainment being the least frequently mentioned (21.3%) yet still representing nearly 4,800 reviews which is a sufficient sample size for aspect-level sentiment analysis.
>
> These aspects were also chosen to align with the structured sub-rating columns already present in the dataset, allowing for direct comparison between passenger-assigned numerical ratings and text-derived sentiment scores:
>
| Aspect (Set C) | Corresponding Skytrax Rating |
|---|---|
| seat | Seat Comfort |
| food | Food & Beverages |
| staff | Cabin Staff Service |
| entertainment | Inflight Entertainment, Wifi & Connectivity |
| ground-service | Ground Service |
>


## **3. Apply Aspect Keywords along with VADER**

In [3]:
# 3.1 Aspect Keyword Dictionary
aspect_keywords = {
    'seat': ['seat', 'legroom', 'comfort', 'recline', 'space', 'cushion', 'comfy', 'spacious', 'stretch'],
    'food': ['food', 'meal', 'drink', 'snack', 'beverage', 'dining', 'menu', 'vegetarian'],
    'staff': ['crew', 'staff', 'attendant', 'stewardess', 'service', 'friendly', 'rude', 'attentive'],
    'ground_service': ['delay', 'late', 'on time', 'punctual', 'schedule', 'depart', 'cancel', 'luggage', 'suitcase',
                       'baggage', 'lost', 'checkin', 'check-in', 'refund', 'booking', ' boarding'],
    'entertainment': ['entertainment', 'screen', 'movie', 'ife', 'wifi', 'music']  # ife stands for In-Flight Entertainment
}

In [4]:
# 3.2 Aspect Sentiment Extraction Function
def extract_aspect_sentiment(text, aspect_keywords):
    if pd.isna(text):
        return {aspect: None for aspect in aspect_keywords}
    
    sentences = sent_tokenize(text)
    aspect_scores = {aspect: [] for aspect in aspect_keywords}
    
    for sentence in sentences:
        for aspect, keywords in aspect_keywords.items():
            if any(keyword in sentence.lower() for keyword in keywords):
                score = analyzer.polarity_scores(sentence)['compound']
                aspect_scores[aspect].append(score)
    
    # Average sentiment per aspect (None if not mentioned)
    result = {}
    for aspect, scores in aspect_scores.items():
        result[aspect] = sum(scores) / len(scores) if scores else None
    
    return result

In [5]:
# 3.3 Test on a Sample Review
sample = df['cleaned_review_C'].iloc[3]
print(sample)
print(extract_aspect_sentiment(sample, aspect_keywords))

never fly adria please favor fly adria route munich pristina july 2019 lost luggage 10 day row despite numerous phone call able locate 11 day later luggage arrived destination completely ruined applying compensation ignored request foolishly booked another flight 345 euro frankfurt pristina september 2019 cancelled flight reason 24 hour departure desperate phone call customer service get anything rerouting compensation etc responded never fly adria disgrace shame adria constantly deceiving customer
{'seat': None, 'food': None, 'staff': -0.9352, 'ground_service': -0.9352, 'entertainment': None}


In [6]:
# 3.4 Apply to Full Dataset
print("Extracting aspect sentiment scores...")
aspect_results = df['cleaned_review_C'].apply(lambda x: extract_aspect_sentiment(x, aspect_keywords))

# Convert to DataFrame
aspect_df = pd.DataFrame(aspect_results.tolist())
aspect_df.columns = [f'aspect_{col}' for col in aspect_df.columns]
print("Done!")

Extracting aspect sentiment scores...


Done!


In [7]:
# 3.5 Check Missing Rate per Aspect
for col in aspect_df.columns:
    missing = aspect_df[col].isnull().sum()
    missing_pct = (missing / len(aspect_df) * 100).round(1)
    print(f"{col}: {missing} missing ({missing_pct}%)")

aspect_seat: 13699 missing (59.6%)
aspect_food: 14171 missing (61.7%)
aspect_staff: 6202 missing (27.0%)
aspect_ground_service: 6438 missing (28.0%)
aspect_entertainment: 18086 missing (78.7%)


> ### **Validating the Aspect Extraction Function**
>
| Aspect | Keyword Presence | Missing Rate |
|---|---|---|
| seat | 40.0% | 59.6% |
| food | 38.0% | 61.7% |
| staff | 70.8% | 27.0% |
| ground-service | 72.0% | 28.0% |
| entertainment | 21.% | 78.7% |
> 
> The missing rate for each aspect column closely mirrors the keyword presence rate measured earlier. Since **keyword presence rate + missing rate ≈ 100%** for each aspect, this confirms that the `extract_aspect_sentiment()` function is behaving as intended where a sentiment score is computed only when the corresponding aspect keyword appears in the review, and `None` is correctly assigned otherwise.
>
> This is not a data quality issue but an expected outcome of the rule-based design — reviews naturally vary in which service aspects they discuss, and the missingness itself carries information (i.e., "this aspect was not mentioned").

In [8]:
# 3.6 Inspect Reviews with All Aspects Missing
all_missing = aspect_df.isnull().all(axis=1).sum()
print(f"Rows with all 5 aspects missing: {all_missing} ({all_missing/len(df)*100:.1f}%)")

Rows with all 5 aspects missing: 473 (2.1%)


> ### **Decision: Accepting Remaining Missing Aspect Coverage**
>
> The proportion of reviews with all five aspects missing is 2.1% (473 reviews), and this is considered an acceptable limitation rather than a problem to fully resolve. The remaining reviews likely involve very short text or highly specific topics outside the scope of the five defined aspects, which fall outside the intended coverage of this rule-based approach.
>
> Further expanding the keyword dictionary to chase a near-zero missing rate would risk diluting the conceptual clarity of each aspect category and introduce diminishing returns. These residual missing values are therefore retained as `None`, consistent with the broader design principle that non-mention is treated as missing information rather than imputed sentiment.

## **4. Save Dataset**

In [9]:
# Merge Aspect Scores into Main DataFrame
df = pd.concat([df, aspect_df], axis=1)
print("Merged successfully!")
df.head()

Merged successfully!


,Airline Name,Verified,Type Of Traveller,Seat Type,Seat Comfort,Cabin Staff Service,Food & Beverages,Ground Service,Inflight Entertainment,Wifi & Connectivity,...,cleaned_review_C,vader_compound,vader_pos,vader_neg,vader_neu,aspect_seat,aspect_food,aspect_staff,aspect_ground_service,aspect_entertainment
0,AB Aviation,True,Solo Leisure,Economy Class,4.0,5.0,4.0,4.0,NaN,NaN,...,pretty decent airline moroni moheli turned pre...,0.9342,0.217,0.000,0.783,NaN,NaN,NaN,0.9468,NaN
1,AB Aviation,True,Solo Leisure,Economy Class,2.0,2.0,1.0,1.0,NaN,NaN,...,good airline moroni anjouan small airline tick...,-0.8244,0.027,0.091,0.882,NaN,NaN,NaN,0.4215,NaN
2,AB Aviation,True,Solo Leisure,Economy Class,2.0,1.0,1.0,1.0,NaN,NaN,...,flight fortunately short anjouan dzaoudzi smal...,0.7569,0.106,0.027,0.867,NaN,NaN,0.8122,NaN,0.8122
3,Adria Airways,False,Solo Leisure,Economy Class,1.0,1.0,NaN,1.0,NaN,NaN,...,never fly adria please favor fly adria route m...,-0.9600,0.034,0.177,0.789,NaN,NaN,-0.9352,-0.9352,NaN
4,Adria Airways,True,Couple Leisure,Economy Class,1.0,1.0,1.0,1.0,1.0,1.0,...,ruined last day holiday book flight airline fr...,0.3027,0.103,0.087,0.810,NaN,NaN,NaN,0.5719,NaN


In [10]:
# Save Aspect Sentiment Scores
import os
os.makedirs('../1_data/processed', exist_ok=True)
out_path = '../1_data/processed/04_aspect_scores.csv'
df.to_csv(out_path, index=False)
print(f"Saved → {out_path}")
print(f"Final shape: {df.shape}")
df.info()

Saved → ../1_data/processed/04_aspect_scores.csv
Final shape: (22980, 26)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22980 entries, 0 to 22979
Data columns (total 26 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Airline Name            22980 non-null  object 
 1   Verified                22980 non-null  bool   
 2   Type Of Traveller       22980 non-null  object 
 3   Seat Type               22980 non-null  object 
 4   Seat Comfort            18763 non-null  float64
 5   Cabin Staff Service     18673 non-null  float64
 6   Food & Beverages        14162 non-null  float64
 7   Ground Service          18292 non-null  float64
 8   Inflight Entertainment  10245 non-null  float64
 9   Wifi & Connectivity     5896 non-null   float64
 10  Value For Money         21806 non-null  float64
 11  Recommended             22980 non-null  int64  
 12  Covid_Period            22980 non-null  int64  
 13  review_length    